# MvDEC 2025 REUTERS scale-balanced fusion diagnostic - seed 42

Runs exactly one preregistered P2 protocol. It keeps the A2 direct View2 head, objective, schedule, data, seed, and evaluator fixed, changing only fusion to per-row L2 normalization before Eq. 4 averaging. This is a scale-sensitivity ablation, not an exact-paper claim.


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this notebook."


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

TRAINING_COMMIT = "9b63446a0dd535a8ab94afc1e5457ab1594fb9ef"
REPO = Path("/kaggle/working/mvdec_fpmax")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", TRAINING_COMMIT], cwd=REPO, check=True)
resolved = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert resolved == TRAINING_COMMIT, (resolved, TRAINING_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
subprocess.run([sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_mvdec_contract.py", "tests/test_data.py", "tests/test_mvdec_preprocessing.py", "tests/test_mvdec_architecture.py", "-k", "l2 or direct_joint_head or loader_contract"], cwd=REPO, check=True)


In [ ]:
PROTOCOL = "mvdec_2025_view2_direct_23_split_l2norm_average_v1"
OUTPUT = Path("/kaggle/working/mvdec-p2-reuters-seed42")
command = [
    sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", "REUTERS",
    "--runs", "1", "--seed", "42", "--protocol", PROTOCOL,
    "--progress-interval", "100",
    "--dataset-root", "external_repos/DEKM/datasets",
    "--public-output-dir", str(OUTPUT),
]
subprocess.run(command, cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pipeline.mvdec_audit", "--root", str(OUTPUT), "--dataset-root", "external_repos/DEKM/datasets"], cwd=REPO, check=True)


In [ ]:
import hashlib
import json
import pickle
import shutil
import numpy as np
import pandas as pd
from sklearn.metrics import normalized_mutual_info_score

run_dirs = sorted((OUTPUT / "reuters" / PROTOCOL).glob("*/seed_42"))
assert len(run_dirs) == 1, run_dirs
run_dir = run_dirs[0]
manifest = json.loads((run_dir / "manifest.json").read_text())
assert manifest["status"] == "complete", manifest
assert manifest["training_commit"] == TRAINING_COMMIT
assert manifest["source_id"] == "dekm_google_drive_reuters_10k"
artifact_path = run_dir / manifest["paths"]["artifact"]
with artifact_path.open("rb") as handle:
    artifact = pickle.load(handle)
assert artifact["fusion_contract"] == "mvdec2025_encoder_l2_normalized_average"
h1 = np.asarray(artifact["h_view1"])
h2 = np.asarray(artifact["h_view2"])
normalize = lambda x: x / np.sqrt(np.maximum(np.sum(x * x, axis=1, keepdims=True), 1e-12))
np.testing.assert_allclose(artifact["h_fused"], (normalize(h1) + normalize(h2)) / 2, rtol=1e-5, atol=1e-6)
summary = {
    "training_commit": TRAINING_COMMIT,
    "protocol_id": PROTOCOL,
    "seed": 42,
    "runtime_seconds": manifest.get("runtime_seconds"),
    "source_sha256": artifact.get("source_sha256"),
    "config_hash": manifest.get("config_hash"),
    **manifest.get("metrics", {}),
    "view1_mean_norm": float(np.linalg.norm(h1, axis=1).mean()),
    "view2_mean_norm": float(np.linalg.norm(h2, axis=1).mean()),
    "fused_mean_norm": float(np.linalg.norm(artifact["h_fused"], axis=1).mean()),
}
pd.DataFrame([summary]).to_csv(OUTPUT / "p2_summary.csv", index=False)
(OUTPUT / "p2_summary.json").write_text(json.dumps(summary, indent=2))
zip_base = "/kaggle/working/mvdec-p2-reuters-seed42"
zip_path = Path(shutil.make_archive(zip_base, "zip", OUTPUT.parent, OUTPUT.name))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = Path(str(zip_path) + ".sha256")
sha_path.write_text(f"{digest}  {zip_path.name}\n")
print(json.dumps(summary, indent=2))
print("ZIP:", zip_path)
print("SHA-256:", digest)
